# Exercise 1 — AIPW, double robustness, and a small TARNet

**Estimated time:** 30 minutes  
**Lecture notation:** observed data $O_i=(W_i,A_i,Y_i)$, outcome regression $Q(w,a)=\mathbb E[Y\mid W=w,A=a]$, propensity score $\pi(a\mid w)=\mathbb P(A=a\mid W=w)$, and ATE $\psi=\mathbb E[Y^1-Y^0]$.

## Learning goals

1. Compare a plug-in estimator, an AIPW estimator, an oracle plug-in estimator, and a simple TARNet.
2. See double robustness in action: AIPW can work when $\widehat Q$ is misspecified if $\widehat\pi$ is correct.
3. Diagnose conceptual mistakes that still produce runnable code, such as omitting a confounder or conditioning on a post-treatment variable.

## Colab instructions

Use **File → Save a copy in Drive** before editing. You can also download the notebook with **File → Download → Download .ipynb**.


In [ ]:
# Setup: this notebook uses only standard Colab packages, except PyTorch, which is usually preinstalled.
# If this cell fails in a local environment, install the missing packages with pip.

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    torch.set_num_threads(1)
except Exception as err:
    raise ImportError("PyTorch is required for the TARNet section. In Colab it is usually already installed.") from err

np.random.seed(123)
torch.manual_seed(123)

plt.rcParams["figure.figsize"] = (8, 4.5)


## 1. A small observational data-generating process

The data are generated so that treatment is confounded by $W$. We also create a variable $M$ that is affected by treatment. That variable is intentionally dangerous: it is predictive of $Y$, but it is **post-treatment**, so adjusting for it changes the causal question.

The true ATE is known because the data-generating mechanism is known. This is the main reason to prefer simulations over benchmark data for a short methods exercise.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def simulate_aipw_data(n=3000, seed=1, exercise=False):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(size=n)
    W2 = rng.normal(size=n)
    W3 = rng.normal(size=n)
    W4 = rng.normal(size=n) if exercise else np.zeros(n)

    if not exercise:
        # Correct propensity model is logistic-linear in these transformed confounder features.
        lin_pi = -0.25 + 0.75 * np.sin(W1) - 0.55 * W2 + 0.35 * (W3 ** 2 - 1)
        pi = np.clip(sigmoid(lin_pi), 0.05, 0.95)
        A = rng.binomial(1, pi)
        tau = 1.5 + 0.6 * W1 - 0.25 * W2
        gW = 1.0 + 1.2 * np.sin(W1) + 0.7 * W2**2 - 0.4 * W3
        M = 0.6 * A + 0.5 * W1 + rng.normal(scale=0.5, size=n)  # post-treatment, not a confounder
        Y = gW + A * tau + rng.normal(scale=1.0, size=n)
        df = pd.DataFrame({"W1": W1, "W2": W2, "W3": W3, "A": A, "M": M, "Y": Y, "pi_true": pi, "tau_true": tau})
    else:
        # New data for the student task: W4 is now a confounder too.
        lin_pi = -0.15 + 0.65 * np.sin(W1) - 0.50 * W2 + 0.35 * (W3 ** 2 - 1) + 0.45 * W4
        pi = np.clip(sigmoid(lin_pi), 0.05, 0.95)
        A = rng.binomial(1, pi)
        tau = 1.2 + 0.8 * (W1 > 0).astype(float) - 0.35 * W2 + 0.25 * W4
        gW = 0.7 + 1.1 * np.sin(W1) + 0.5 * W2**2 - 0.25 * W3 + 0.4 * W4
        M = 0.9 * A + 0.6 * W1 - 0.3 * W4 + rng.normal(scale=0.6, size=n)  # post-treatment
        Y = gW + A * tau + 0.4 * M + rng.normal(scale=1.0, size=n)
        # Note: because M is post-treatment and also affects Y here, conditioning on M targets a controlled/direct-effect-like object, not the total ATE.
        total_tau = tau + 0.4 * 0.9  # total effect includes the mediated component A -> M -> Y
        df = pd.DataFrame({"W1": W1, "W2": W2, "W3": W3, "W4": W4, "A": A, "M": M, "Y": Y, "pi_true": pi, "tau_true": total_tau, "tau_direct": tau})
    return df


def Q_true_example(df, a):
    # E[Y | W, A=a] for the first, non-exercise DGP.
    W1, W2, W3 = df["W1"].values, df["W2"].values, df["W3"].values
    tau = 1.5 + 0.6 * W1 - 0.25 * W2
    gW = 1.0 + 1.2 * np.sin(W1) + 0.7 * W2**2 - 0.4 * W3
    return gW + a * tau


def pi_features_example(df):
    # Correct features for the first DGP's propensity score.
    return pd.DataFrame({
        "sin_W1": np.sin(df["W1"].values),
        "W2": df["W2"].values,
        "W3_sq_minus_1": df["W3"].values ** 2 - 1,
    })


def summarize_true_effect(df):
    return df["tau_true"].mean()

# Training data and a large reference population for a stable approximation to the true ATE.
df = simulate_aipw_data(n=2500, seed=10, exercise=False)
ref = simulate_aipw_data(n=100_000, seed=999, exercise=False)
psi_true = summarize_true_effect(ref)

print(df.head())
print(f"Approximate true ATE psi = {psi_true:.3f}")
print(f"Treatment prevalence = {df['A'].mean():.3f}")


## 2. Completed example: plug-in with a misspecified $Q$

The plug-in estimator uses

$$
\widehat\psi_{\rm plug}=\frac{1}{n}\sum_i \{\widehat Q(W_i,1)-\widehat Q(W_i,0)\}.
$$

Here we deliberately use a linear regression that omits nonlinearities. The code is correct, but the model class is wrong.


In [ ]:
def fit_plugin_linear(df, W_cols):
    X = df[W_cols + ["A"]]
    y = df["Y"].values
    model = make_pipeline(StandardScaler(), LinearRegression())
    model.fit(X, y)

    X1 = df[W_cols].copy(); X1["A"] = 1
    X0 = df[W_cols].copy(); X0["A"] = 0
    Q1_hat = model.predict(X1[W_cols + ["A"]])
    Q0_hat = model.predict(X0[W_cols + ["A"]])
    Q_A_hat = np.where(df["A"].values == 1, Q1_hat, Q0_hat)
    psi_hat = np.mean(Q1_hat - Q0_hat)
    return {"psi": psi_hat, "Q1": Q1_hat, "Q0": Q0_hat, "Q_A": Q_A_hat, "model": model}

plugin_bad = fit_plugin_linear(df, W_cols=["W1", "W2", "W3"])
print(f"Misspecified plug-in estimate = {plugin_bad['psi']:.3f}")


## 3. Completed example: AIPW with misspecified $Q$ but correct $\pi$

The AIPW score for the ATE is

$$
\widehat\varphi_i = \widehat Q(W_i,1)-\widehat Q(W_i,0) +
\left\{\frac{A_i}{\widehat\pi(1\mid W_i)}-\frac{1-A_i}{\widehat\pi(0\mid W_i)}\right\}
\{Y_i-\widehat Q(W_i,A_i)\}.
$$

Then $\widehat\psi_{\rm AIPW}=n^{-1}\sum_i\widehat\varphi_i$. The estimated standard error is the empirical standard deviation of the influence-function scores divided by $\sqrt n$.


In [ ]:
def fit_propensity_correct_for_example(df):
    X_pi = pi_features_example(df)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(X_pi, df["A"].values)
    pi1_hat = model.predict_proba(X_pi)[:, 1]
    pi1_hat = np.clip(pi1_hat, 0.02, 0.98)
    return pi1_hat, model


def aipw_ate(Y, A, Q0_hat, Q1_hat, pi1_hat):
    pi1_hat = np.clip(pi1_hat, 0.02, 0.98)
    Q_A_hat = np.where(A == 1, Q1_hat, Q0_hat)
    phi = (Q1_hat - Q0_hat) + (A / pi1_hat - (1 - A) / (1 - pi1_hat)) * (Y - Q_A_hat)
    psi = phi.mean()
    sd_phi = phi.std(ddof=1)
    se = sd_phi / np.sqrt(len(Y))
    return {"psi": psi, "sd_phi": sd_phi, "se": se, "phi": phi}

pi1_hat, pi_model = fit_propensity_correct_for_example(df)
aipw_good_pi = aipw_ate(
    Y=df["Y"].values,
    A=df["A"].values,
    Q0_hat=plugin_bad["Q0"],
    Q1_hat=plugin_bad["Q1"],
    pi1_hat=pi1_hat,
)

print(f"AIPW estimate = {aipw_good_pi['psi']:.3f}")
print(f"Influence-function SD = {aipw_good_pi['sd_phi']:.3f}")
print(f"SE = {aipw_good_pi['se']:.3f}")
print(f"Approx. 95% CI = [{aipw_good_pi['psi'] - 1.96*aipw_good_pi['se']:.3f}, {aipw_good_pi['psi'] + 1.96*aipw_good_pi['se']:.3f}]")


## 4. Completed example: oracle plug-in

The oracle plug-in is not available in real data because it uses the true conditional mean. It is included here to anchor the comparison.


In [ ]:
Q1_oracle = Q_true_example(df, a=1)
Q0_oracle = Q_true_example(df, a=0)
psi_oracle_plugin = np.mean(Q1_oracle - Q0_oracle)
print(f"Oracle plug-in estimate = {psi_oracle_plugin:.3f}")


## 5. Completed example: simple TARNet

TARNet learns a shared representation of $W$ and then two outcome heads, one for $A=0$ and one for $A=1$. In this toy notebook, it is used as a flexible plug-in estimator:

$$
\widehat\psi_{\rm TARNet}=\frac{1}{n}\sum_i \{\widehat Q_1(W_i)-\widehat Q_0(W_i)\}.
$$

This is **not** doubly robust. It is a flexible outcome-regression estimator.


In [ ]:
class SimpleTARNet(nn.Module):
    def __init__(self, p, hidden=32):
        super().__init__()
        self.repr = nn.Sequential(
            nn.Linear(p, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.head0 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        h = self.repr(x)
        return self.head0(h).squeeze(-1), self.head1(h).squeeze(-1)


def fit_tarnet(df, W_cols, epochs=150, lr=1e-3, seed=123, verbose=False):
    torch.manual_seed(seed)
    scaler = StandardScaler()
    W_np = scaler.fit_transform(df[W_cols].values).astype("float32")
    A_np = df["A"].values.astype("float32")
    Y_np = df["Y"].values.astype("float32")

    W = torch.tensor(W_np)
    A = torch.tensor(A_np)
    Y = torch.tensor(Y_np)

    net = SimpleTARNet(p=W.shape[1], hidden=32)
    opt = optim.Adam(net.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        q0, q1 = net(W)
        qA = A * q1 + (1 - A) * q0
        loss = loss_fn(qA, Y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if verbose and epoch % 100 == 0:
            print(epoch, float(loss.detach()))

    with torch.no_grad():
        q0, q1 = net(W)
    Q0_hat = q0.numpy()
    Q1_hat = q1.numpy()
    return {"psi": np.mean(Q1_hat - Q0_hat), "Q0": Q0_hat, "Q1": Q1_hat, "net": net, "scaler": scaler}

tarnet = fit_tarnet(df, W_cols=["W1", "W2", "W3"], epochs=150, lr=1e-3)
print(f"Simple TARNet plug-in estimate = {tarnet['psi']:.3f}")


## 6. Compare estimates

For AIPW, the error bar is the influence-function 95% interval. For plug-in and TARNet, the plot shows point estimates only, because naive regression standard errors do not account for nuisance estimation in a valid causal way.


In [ ]:
results = pd.DataFrame({
    "estimator": ["true psi", "plug-in: misspecified Q", "AIPW: misspecified Q, correct pi", "oracle plug-in", "simple TARNet"],
    "estimate": [psi_true, plugin_bad["psi"], aipw_good_pi["psi"], psi_oracle_plugin, tarnet["psi"]],
    "se": [np.nan, np.nan, aipw_good_pi["se"], np.nan, np.nan],
})
results


In [ ]:
fig, ax = plt.subplots()
ypos = np.arange(len(results))
ax.scatter(results["estimate"], ypos)

# Add 95% CI only where an IF-based SE is available.
for j, row in results.iterrows():
    if np.isfinite(row["se"]):
        ax.errorbar(row["estimate"], j, xerr=1.96 * row["se"], fmt="none", capsize=4)

ax.axvline(psi_true, linestyle="--", label="true psi")
ax.set_yticks(ypos)
ax.set_yticklabels(results["estimator"])
ax.set_xlabel("ATE estimate")
ax.set_title("AIPW corrects a misspecified outcome regression when pi is correct")
ax.legend()
plt.show()


# Student task: new data, same ideas

You now receive a new simulated dataset. Your task is to repeat the comparison.

The intentionally tricky part is not syntax. The tricky part is deciding what belongs in $W$.

**Available variables**

- `W1`, `W2`, `W3`, `W4`: baseline variables measured before treatment.
- `A`: treatment.
- `M`: post-treatment variable. It is predictive of $Y$, but it is not a baseline confounder.
- `Y`: outcome.

## Discussion trap

A model using `M` may predict $Y$ better. But if you condition on `M`, are you estimating the total effect of $A$ on $Y$, or a different controlled/direct-effect-like object?


In [ ]:
df_ex = simulate_aipw_data(n=3000, seed=2026, exercise=True)
ref_ex = simulate_aipw_data(n=100_000, seed=2027, exercise=True)
psi_true_ex = summarize_true_effect(ref_ex)

print(df_ex.head())
print(f"Approximate true total ATE in the new data = {psi_true_ex:.3f}")
print(f"Treatment prevalence = {df_ex['A'].mean():.3f}")


## Task 1 — Choose the adjustment variables

Fill the lists below. Some choices are defensible for prediction but wrong for the total ATE. Some choices produce runnable code but omit confounding.

Try at least two choices and compare:

1. A plausible causal choice.
2. A prediction-driven but causally suspicious choice.


In [ ]:
# TODO: choose baseline adjustment variables for the outcome model Q(w,a).
# Candidate variables: "W1", "W2", "W3", "W4", "M".
# Hint: M is post-treatment.
W_for_Q = ["W1", "W2", "W3", "W4"]  # <-- edit this

# TODO: choose variables for the propensity score pi(1|w).
# Should a post-treatment variable be used to model treatment assignment?
W_for_pi = ["W1", "W2", "W3", "W4"]  # <-- edit this

print("Outcome adjustment variables:", W_for_Q)
print("Propensity adjustment variables:", W_for_pi)


## Task 2 — Fit a misspecified plug-in estimator

This code intentionally uses a simple linear model. You control the conceptual choice of variables.


In [ ]:
plugin_ex = fit_plugin_linear(df_ex, W_cols=W_for_Q)
print(f"Plug-in estimate with your Q variables = {plugin_ex['psi']:.3f}")


## Task 3 — Fit a propensity model

The propensity score is $\varpi(w)=\pi(1\mid w)=\mathbb P(A=1\mid W=w)$. A flexible or correctly specified propensity model is what gives AIPW a chance to correct a bad $Q$.


In [ ]:
def fit_propensity_for_student(df, W_cols):
    # This model is flexible enough for the exercise, but it cannot fix a conceptually wrong adjustment set.
    model = make_pipeline(
        PolynomialFeatures(degree=2, include_bias=False),
        StandardScaler(),
        LogisticRegression(max_iter=3000),
    )
    model.fit(df[W_cols], df["A"].values)
    pi1_hat = model.predict_proba(df[W_cols])[:, 1]
    return np.clip(pi1_hat, 0.02, 0.98), model

pi_ex, pi_model_ex = fit_propensity_for_student(df_ex, W_for_pi)
print(pd.Series(pi_ex).describe())


## Task 4 — Complete the AIPW score

Complete the two lines marked `TODO`. This is a conceptual check: the first line asks which counterfactual prediction corresponds to the observed treatment, and the second asks for the AIPW residual correction.


In [ ]:
def aipw_score_student(Y, A, Q0_hat, Q1_hat, pi1_hat):
    pi1_hat = np.clip(pi1_hat, 0.02, 0.98)

    # TODO 1: observed-treatment outcome prediction Q(W_i, A_i).
    # Correct form uses Q1_hat for treated units and Q0_hat for controls.
    Q_A_hat = np.where(A == 1, Q1_hat, Q0_hat)  # <-- edit if you want to test alternatives

    # TODO 2: AIPW pseudo-outcome / influence-function score for the ATE.
    # The residual correction must use both treatment arms with signs + for A=1 and - for A=0.
    phi = (Q1_hat - Q0_hat) + (A / pi1_hat - (1 - A) / (1 - pi1_hat)) * (Y - Q_A_hat)  # <-- edit if needed

    psi = phi.mean()
    sd_phi = phi.std(ddof=1)
    se = sd_phi / np.sqrt(len(Y))
    return psi, sd_phi, se, phi

psi_aipw_ex, sd_phi_ex, se_aipw_ex, phi_ex = aipw_score_student(
    Y=df_ex["Y"].values,
    A=df_ex["A"].values,
    Q0_hat=plugin_ex["Q0"],
    Q1_hat=plugin_ex["Q1"],
    pi1_hat=pi_ex,
)
print(f"AIPW estimate = {psi_aipw_ex:.3f}")
print(f"Influence-function SD = {sd_phi_ex:.3f}")
print(f"SE = {se_aipw_ex:.3f}")


## Task 5 — Compare against a flexible plug-in TARNet

Here the architecture is provided. Your task is again conceptual: decide what goes into $W$.


In [ ]:
tarnet_ex = fit_tarnet(df_ex, W_cols=W_for_Q, epochs=150, lr=1e-3, seed=2026)
print(f"Simple TARNet plug-in estimate with your Q variables = {tarnet_ex['psi']:.3f}")


In [ ]:
student_results = pd.DataFrame({
    "estimator": ["true total ATE", "plug-in", "AIPW", "simple TARNet"],
    "estimate": [psi_true_ex, plugin_ex["psi"], psi_aipw_ex, tarnet_ex["psi"]],
    "se": [np.nan, np.nan, se_aipw_ex, np.nan],
})
student_results


In [ ]:
fig, ax = plt.subplots()
ypos = np.arange(len(student_results))
ax.scatter(student_results["estimate"], ypos)
for j, row in student_results.iterrows():
    if np.isfinite(row["se"]):
        ax.errorbar(row["estimate"], j, xerr=1.96 * row["se"], fmt="none", capsize=4)
ax.axvline(psi_true_ex, linestyle="--", label="true total ATE")
ax.set_yticks(ypos)
ax.set_yticklabels(student_results["estimator"])
ax.set_xlabel("ATE estimate")
ax.set_title("Your choices: adjustment set and AIPW score")
ax.legend()
plt.show()


## Questions for discussion

1. What changed when you included `M` in the outcome model?
2. What changed when you included `M` in the propensity model?
3. Did AIPW rescue the misspecified plug-in estimator? Under which variable choice?
4. Why is TARNet not automatically doubly robust?
5. If the plug-in and TARNet predict $Y$ well, does that imply the ATE is well estimated?
